# Figure 1

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import os
import time

import plotly.offline as pyo
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white" # https://medium.com/plotly/introducing-plotly-py-theming-b644109ac9c7

directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\Final datasets used in article'
os.chdir(directory)

remove_UA_RU=False
smooth=False
smooth_time=3
Filter=False


In [ ]:
DFfreq = pd.read_csv(r'pivotUkraine_SUM_excelInput_weekly.csv') #, index_col=0, sep=';'
#Load frequency share pivoted df
DFfreq.set_index(DFfreq.columns[0], inplace=True)
DFfreq.head(3)

In [ ]:
if remove_UA_RU==True:
    DFfreq = DFfreq.drop(index=['Ukrainian', 'Russian'])

          ###### 1-Mean freq of language
#What is the average freq of each language>?
freqRowMeans = DFfreq.mean(axis=1)

          ###### 2-Over and underexplression
#Mean per language
DFresultNonLog = DFfreq.sub(freqRowMeans, axis=0)
DFresultNonLog = DFresultNonLog.div(freqRowMeans, axis=0)

          ###### 3-Log Over and underexplression
#Put the last result to log scale (take into account neg values)
DFresultLog = DFresultNonLog.applymap(lambda x: 0 if x == 0 else (np.log10(x * 100000) if x > 0 else -np.log10(-x * 100000)))

#Smooth data
min_periods=1
if smooth:
    DFresultLog = DFresultLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    DFresultNonLog = DFresultNonLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    DFfreq = DFfreq.T.rolling(window=smooth_time, min_periods=min_periods).mean().T


# Visualization

In [ ]:
#Unpivoting table from wide to long format
AttentionLog=DFresultLog.reset_index().melt(id_vars=['language'], var_name='Date', value_name='ExpressionLog')
AttentionNonLog=DFresultNonLog.reset_index().melt(id_vars=['language'], var_name='Date', value_name='Expression')
Freq=DFfreq.reset_index().melt(id_vars=['language'], var_name='Date', value_name='FractionRaw')

# Merge freq and attention share dfs
merged_df = Freq.merge(AttentionLog, on=['language', 'Date'], how='inner')
merged_df = merged_df.merge(AttentionNonLog, on=['language', 'Date'], how='inner')
#display(merged_df)
#print(merged_df.describe())
merged_df['FractionLog'] = np.log10(merged_df['FractionRaw'])+7.0 #Cant do size with neg values

merged_df['Fraction']= merged_df['FractionRaw'].replace(0, np.nan)
merged_df['FractionLog']=np.log10(merged_df.Fraction)
#print(merged_df.describe())

merged_df['FractionLog']= (merged_df['FractionLog'])+7.5 #Cant do size with neg values
merged_df['FractionLog']=merged_df['FractionLog'].fillna(0)
merged_df['FractionRaw']=merged_df['FractionRaw'].fillna(0)

manual_order = [
     'Ukrainian', 'Russian', 'Arabic', 'Portuguese', 'Catalan', 'Korean', 'Persian', 'Turkish', 'Indonesian', 'Urdu', 'Vietnamese',
    'Serbian', 'Estonian', 'Romanian', 'Greek', 'Hungarian', 'Polish', 'Swedish', 'Czech', 'Spanish', 
    'Danish', 'English', 'Dutch', 'Norwegian','Finnish', 'French', 'Italian', 'German'
]
'''
#NEW ORDER
manual_order = [
    'Ukrainian', 'Russian', 'Urdu', 'Vietnamese', 'Serbian', 'Estonian', 'Hungarian',
    'Turkish', 'Arabic', 'Korean', 'Catalan', 'Indonesian', 'Portuguese',
    'Polish', 'Czech', 'Norwegian', 'Finnish', 'Dutch', 'English', 'Swedish', 'Danish',
    'French', 'Italian', 'German', 'Spanish', 'Greek', 'Romanian'
]
'''

manual_order.reverse()

merged_df['language'] = pd.Categorical(merged_df['language'], categories=manual_order, ordered=True)
merged_df = merged_df.sort_values(['language', 'Date'])


# Plot

In [ ]:
#NOLOG
import plotly.express as px
import numpy as np

# Extract the ExpressionLog column from merged_df, removing NaN values
expression_log_clean = merged_df['Expression'].dropna()

# Create the histogram with the bars colored gray
fig = px.histogram(
    expression_log_clean, 
    nbins=170,  # Number of bins for the histogram
    title='_with-UA-RU_Distribution of Over Underexpression (Expression)',
    histnorm='density',  # Normalize the histogram to get a density plot
    color_discrete_sequence=['gray']  # Set histogram bars color to gray
)
fig.update_traces(marker_line_width=0)

# Customize the plot
fig.update_layout(
    xaxis_title="Log Frequency", 
    yaxis_title="Density",
    bargap=0,
    bargroupgap=0,
    font=dict(size=14),
    title_font=dict(size=16),
    template="simple_white",  # Cleaner white background
    showlegend=False,
    margin=dict(l=50, r=50, t=60, b=50),  # Adjust margins for a cleaner layout
)

# Show the plot
fig.show()


In [ ]:
#SMALL
import plotly.express as px
import numpy as np

# Extract the ExpressionLog column from merged_df, removing NaN values
expression_log_clean = merged_df['ExpressionLog'].dropna()

# Create the histogram with the bars colored gray
fig = px.histogram(
    expression_log_clean, 
    nbins=170,  # Number of bins for the histogram
    title='_with-UA-RU_Distribution of Over Underexpression (ExpressionLog)',
    histnorm='density',  # Normalize the histogram to get a density plot
    color_discrete_sequence=['gray']  # Set histogram bars color to gray
)
fig.update_traces(marker_line_width=0)

# Customize the plot
fig.update_layout(
    xaxis_title="Log Frequency", 
    yaxis_title="Density",
    bargap=0,
    bargroupgap=0,
    font=dict(size=14),
    title_font=dict(size=16),
    template="simple_white",  # Cleaner white background
    showlegend=False,
    margin=dict(l=50, r=50, t=60, b=50),  # Adjust margins for a cleaner layout
)

# Show the plot
fig.show()


In [ ]:
#LARGE FOR SAVING
import plotly.express as px
import numpy as np

# Flatten the DataFrame to get all values in a single array for plotting
all_values = DFresultLog.values.flatten()

# Create the histogram with the bars colored gray
fig = px.histogram(
    all_values, 
    nbins=170,  # Number of bins for the histogram
    #title='_With-UA-RU__Distribution of Over Underexpression',
    histnorm='density',  # Normalize the histogram to get a density plot
    color_discrete_sequence=['gray']  # Set histogram bars color to gray
)
fig.update_traces(marker_line_width=0)

# Customize the plot
fig.update_layout(
    xaxis_title="", 
    yaxis_title="",
    bargap=0,
    bargroupgap=0,
    font=dict(size=28),  # Increase the base font size
    title_font=dict(size=20),  # Increase title font size
    template="simple_white",  # Cleaner white background
    showlegend=False,
    margin=dict(l=50, r=50, t=60, b=50),  # Adjust margins for a cleaner layout
    width=2127,  # Set width in pixels for 180 mm at 300 DPI
    height=700  # Set height; adjust as needed
)

# Show the plot
fig.show()


In [ ]:
import os
from subprocess import call

# Set the figure directory and filename criteria
sublocation = f'/Visualizations/Fig.1-2_Heatworm/within-language/SUMSmooth{smooth_time}w/Distribution/'
# Define the plot name
plotname = 'Distribution'  # Replace with your desired plot name

# Choose the name based on criteria
if remove_UA_RU:
    name = f'{plotname}_Smooth_NO-UA-RU' if smooth else f'{plotname}_NO-Smooth_NO-UA-RU'
else:
    name = f'{plotname}_Smooth_with-UA-RU' if smooth else f'{plotname}_NO-Smooth_with-UA-RU'

# Define the base directory for saving files
directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter'
os.chdir(directory)

# Make the main figure directory if not present
main_figure_path = directory + sublocation
os.makedirs(main_figure_path, exist_ok=True)

# Define output formats and create their subdirectories
formats = ['pdf', 'jpeg', 'svg']
for fmt in formats:
    os.makedirs(main_figure_path + fmt + "/", exist_ok=True)  # Use forward slash for consistency

# Save the figure in different formats
for fmt in formats:
    file_path = main_figure_path + fmt + "/" + "_" + name + "." + fmt  # Ensure consistent path formatting
    fig.write_image(file_path, format=fmt, engine="kaleido")
    print(f"Saved {fmt.upper()} to: {file_path}")

# Convert PDF to EPS if needed
# call(["pdf2ps", main_figure_path + "pdf" + "/" + "_" + name + ".pdf", main_figure_path + "eps" + "/" + "_" + name + ".eps"])


In [ ]:
if remove_UA_RU==True:
    setheight=26
else:
    setheight=28

def add_custom_annotation(fig, text, date, height,color):
    # Add line
    fig.add_shape(
        type='line',
        x0=date,
        y0=-2,
        x1=date,
        y1=setheight, #height,
        line=dict(color=color, width=0.6, dash="dot")
    )

    # Add annotation
    fig.add_annotation(
        x=date,
        y=height,
        xref='x',
        yref='y',
        text=text,
        showarrow=False,
        ax=0,
        ay=-50
    )


In [ ]:
#old colroscales
# 0.1 and less is gray, more is getting colors
custom_colorscale = [
    [0, '#013b50'], # Min color (blue)
    [0.1706, '#1b83a9'],    # Middle color (gray)
    [0.2562, '#B6B6A8'],    # Middle color (gray)
    [0.4272, '#B6B6A8'],    # Middle color (gray)
    [0.5983, '#B6B6A8'],    # Middle color (gray)
    [0.6838, '#a41a6b'],    # Middle color (gray)
    [1, '#4d012d']  # Max color (red)
]

# 0.1 and less is gray, more is getting colors BUT red part is more nuanced (dark red comes later)
custom_colorscale = [
    [0, '#013b50'],    # Min color (dark blue)
    [0.1706, '#1b83a9'],    # Middle color (light blue)
    [0.2562, '#B6B6A8'],    # Middle color (gray)
    [0.4272, '#B6B6A8'],    # Middle color (gray)
    [0.5983, '#B6B6A8'],    # Middle color (gray)
    [0.6838, '#a41a6b'],    # Starting pink (transition)
    [0.85, '#d20e5d'],      # More intense pink (higher overexpression)
    [0.95, '#9f0116'],      # Strong red (near overexpression peak)
    [1, '#4d012d']  # Max color (deep red)
]

######### 
# Create the custom colorscale
custom_colorscale = [
    [0, '#013b50'],    # Min color (dark blue)
    [0.0851, '#1b83a9'],  #-4  # Middle color (light blue)
    [0.1706, '#B6B6A8'],  #-3  # Middle color (gray)
    [0.4272, '#B6B6A8'],  #0  # Middle color (gray)
    [0.6838, '#B6B6A8'],  #3  # Middle color (gray)
    [0.7694, '#d28db5'],  #4  # Starting pink (transition)
    [0.8549, '#a41a6b'],  #5    # More intense pink (higher overexpression)
    [0.9404, '#2a0313'],  #6   # Strong red (near overexpression peak)
    [1, '#2a0313']  # Max color (deep red)
]


In [ ]:
merged_df['ExpressionLog'].describe()

In [ ]:
def calculate_normalized_position(value, min_val, max_val):
    normalized_position = (value - min_val) / (max_val - min_val)
    return normalized_position

# Example usage
min_val = -4.995016
max_val = 6.696558
value = 6


normalized_position = calculate_normalized_position(value, min_val, max_val)
print(f"Normalized position of {value} is: {normalized_position:.4f}")


In [ ]:
#### SET COLORS
# Get summary statistics from your dataframe column
stats = merged_df['ExpressionLog'].describe()
min_val = stats['min']
max_val = stats['max']

# Define all color points together as (value, color) pairs.
# The first and last values are the data's min and max, respectively.
color_points = [
    (min_val, '#013b50'),  # Min color (dark blue)
    (-4, '#1b83a9'),       # threshold -4 (light blue)
    (-3, '#B6B6A8'),       # threshold -3 (gray)
    (0,  '#B6B6A8'),       # threshold  0 (gray)
    (3,  '#B6B6A8'),       # threshold  3 (gray)
    (4,  '#d28db5'),       # threshold  4 (starting pink)
    (5,  '#a41a6b'),       # threshold  5 (more intense pink)
    (6,  '#2a0313'),       # threshold  6 (strong red)
    (max_val, '#2a0313')   # Max color (deep red)
]

# Create the custom colorscale by computing normalized positions
custom_colorscale = []
for value, color in color_points:
    norm_pos = (value - min_val) / (max_val - min_val)
    custom_colorscale.append([norm_pos, color])

# Print the complete colorscale along with the original values for clarity
print("Custom Colorscale:")
for (val, color), (norm, _) in zip(color_points, custom_colorscale):
    print(f"Value {val}: Normalized Position {norm:.4f} -> Color {color}")

# custom_colorscale is now ready to be used in your Plotly plot
#print(custom_colorscale)


In [ ]:
# SMALL VERSION
import plotly.io as pio
import plotly.graph_objs as go
import pandas as pd

# Set the template
pio.templates.default = "plotly_white"

# Create hovertext list
hovertext = [f"Expression: {first}<br>ExpressionLog: {second}" for first, second in zip(merged_df['Expression'], merged_df['ExpressionLog'])]

# Create the scatter plot using the ExpressionLog values for color
fig = go.Figure(data=[go.Scatter(
    x=merged_df['Date'],
    y=merged_df['language'],
    marker_size=merged_df['FractionLog'],
    marker_color=merged_df['ExpressionLog'],  # Directly use ExpressionLog values for color
    hovertext=hovertext,
    mode='markers',
    marker=dict(
        sizemode='diameter',
        sizeref=0.60,
        sizemin=1,
        opacity=1,
        symbol='line-ns',
        line=dict(
            color=merged_df['ExpressionLog'],  # Directly use ExpressionLog values for line color
            colorscale=custom_colorscale,  # Set the custom colorscale
            width=2
        ),
        colorscale=custom_colorscale,  # Set the custom colorscale
        #cmin=-4.5,  # Set the min value for the color scale
        #cmax=4.5,   # Set the max value for the color scale
        colorbar=dict(  # Add the colorbar settings
            #title='Expression Log',
            orientation='h',  # Horizontal orientation
            thickness=10,
            len=0.4,  # Length of colorbar as a fraction of the plot width
            x=0.5,  # Centered in the x-axis
            xanchor='center',  # Anchor the x position
            y=-0.3,  # Position below the plot
            ticktext=["-10%", "-1%", "-0.1%", "0%", "+0.1%", "+1%", "+10%", "+100%","+1000%"],  # Custom tick values
            tickvals=[-4, -3, -2, 0, 2, 3, 4,5,6],  # Corresponding text for the ticks
            ticks='outside',  # Optional: position ticks outside the colorbar
            tickangle=0 )
        )
    )])

# Add custom annotations (assuming the add_custom_annotation function is defined)
add_custom_annotation(fig, '', '2009-01-5', 26, 'black')
add_custom_annotation(fig, '', '2012-06-08', 26, 'black')
add_custom_annotation(fig, '', '2014-02-27', 26, 'black')
add_custom_annotation(fig, '', '2022-02-24', 26, 'black')

# Generate yearly ticks
yearly_ticks = pd.date_range(start='2008-01-01', end='2023-12-31', freq='YS')
ticktext = ['' for _ in range(len(yearly_ticks))]
for i, date in enumerate(yearly_ticks):
    if date.year >= 2009 and (date.year - 2009) % 2 == 0:
        ticktext[i] = str(date.year)

# Update layout with no margins and more transparent yearly lines
fig.update_layout(
    margin=dict(l=0, r=10, t=0, b=0),  # Remove margins
    yaxis=dict(
        tickmode='linear',
        dtick=1,
        showgrid=True,
        gridcolor='rgba(175,175,175,0.2)',  # Make grid lines more transparent
        range=[-1, 30]
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=yearly_ticks.strftime('%Y-%m-%d').tolist(),
        ticktext=ticktext,
        range=['2008-10-01', '2023-05-30'],
        showgrid=True,
        gridcolor='rgba(175,175,175,0.2)'  # Make grid lines more transparent
    ),
    xaxis_title='',
    yaxis_title='',
)

#fig.show()


In [ ]:
#LARGE VERSION
pio.templates.default = "plotly_white"  # Set the template


##### ##### ##### CALCULATE BAR HEIGHT

### SET HEIGHTS AND CALCULATE MAX BAR HEIGHT
height=1350
width=2000

# Determine the number of unique language rows
n_languages = merged_df['language'].nunique()

# Calculate available vertical pixels per row (using the figure height of 1350)
row_height_pixels = height / n_languages

# Target fraction of the row height for the largest marker; using 0.56 gives ~25 pixels if row height is ~45 pixels.
target_fraction_of_row = 0.52  
target_marker_pixels = row_height_pixels * target_fraction_of_row

# Compute sizeref so that the maximum marker (FractionLog.max()) becomes the target pixel height
computed_sizeref = float(merged_df['FractionLog'].max() / target_marker_pixels)

print("Computed sizeref:", computed_sizeref)

##### ##### ##### CALCULATE BAR WIDTH

# 1) How many *distinct* x-positions do you have?
unique_dates = pd.to_datetime(merged_df['Date']).sort_values().unique()
n_slots = len(unique_dates)

# 2) Your figure width in pixels
fig_width_px = width  # e.g. 2000

# 3) Available pixels per slot
slot_px = fig_width_px / n_slots

# 4) Pick a fraction so you don’t completely fill each slot
fraction = 1
target_line_width_px = slot_px * fraction


##### ##### ##### PLOT


# Create hovertext list
hovertext = [f"Fraction: {count}<br>ExpressionLog: {rank}" for count, rank in zip(merged_df['Expression'], merged_df['ExpressionLog'])]

# Create the scatter plot
fig = go.Figure(data=[go.Scatter(
    x=merged_df['Date'],
    y=merged_df['language'],
    marker_size=merged_df['FractionLog'],
    marker_color=merged_df['ExpressionLog'],
    hovertext=hovertext,
    mode='markers',
    marker=dict(
        sizemode='diameter',
        sizeref=computed_sizeref,#0.20, #
        sizemin=1,
        opacity=1,
        symbol='line-ns',
        line=dict(
            color=merged_df['ExpressionLog'],  # Directly use ExpressionLog values for line color
            colorscale=custom_colorscale,  # Set the custom colorscale
            width=target_line_width_px
        ),
        colorscale=custom_colorscale,  # Set the custom colorscale
        #cmin=-4.5,  # Set the min value for the color scale
        #cmax=4.5,   # Set the max value for the color scale
            colorbar=dict(  # Add the colorbar settings
            #title='Expression Log',
            orientation='h',  # Horizontal orientation
            thickness=50,
            len=0.6,  # Length of colorbar as a fraction of the plot width
            x=0.5,  # Centered in the x-axis
            xanchor='center',  # Anchor the x position
            y=-0.3,  # Position below the plot
            ticktext=["-10%", "-1%", "-0.1%", "0%", "+0.1%", " 1%", " 10%", " 100%","    1000%"],  # Custom tick values
            tickvals=[-4, -3, -2, 0, 2, 3, 4,5,6],  # Corresponding text for the ticks
                    ticks='outside',  # Optional: position ticks outside the colorbar
                tickangle=0
        )
    )
)])

# Add custom annotations
add_custom_annotation(fig, '', '2009-01-5', 26, 'black')
add_custom_annotation(fig, '', '2012-06-08', 26, 'black')
add_custom_annotation(fig, '', '2014-02-27', 26, 'black')
add_custom_annotation(fig, '', '2022-02-24', 26, 'black')

# Generate yearly ticks
yearly_ticks = pd.date_range(start='2008-01-01', end='2023-12-31', freq='YS')
ticktext = ['' for _ in range(len(yearly_ticks))]
for i, date in enumerate(yearly_ticks):
    if date.year >= 2009 and (date.year - 2009) % 1 == 0:
        ticktext[i] = str(date.year)

# Update layout with no margins and more transparent yearly lines
fig.update_layout(
    height=height,
    width=width,
    margin=dict(l=0, r=10, t=0, b=0),  # Remove margins
    yaxis=dict(
        tickmode='linear',
        dtick=1,
        showgrid=True,
        gridcolor='rgba(175,175,175,0.2)',  # Make grid lines more transparent
        range=[-1, 30]
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=yearly_ticks.strftime('%Y-%m-%d').tolist(),
        ticktext=ticktext,
        tickangle=270,
        range=['2008-10-01', '2023-05-30'],
        showgrid=True,
        gridcolor='rgba(175,175,175,0.2)'  # Make grid lines more transparent
    ),
    xaxis_title='',
    yaxis_title='',
    font=dict(size=32),
)

fig.show()




In [ ]:
import os
from subprocess import call
plotname = 'Heatworm'  # Replace with your desired plot name

# Set the figure directory and filename criteria
sublocation = f'/Visualizations/Fig.1-2_Heatworm/within-language/SUMSmooth{smooth_time}w/'
#sublocation = f'/Visualizations/Fig.1-2_Heatworm/within-language/SUM_no_smooth/'

# Choose the name based on criteria
if remove_UA_RU:
    name = f'{plotname}_Smooth_NO-UA-RU' if smooth else f'{plotname}_NO-Smooth_NO-UA-RU'
else:
    name = f'{plotname}_Smooth_with-UA-RU' if smooth else f'{plotname}_NO-Smooth_with-UA-RU'

# Define the base directory for saving files
directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter'
os.chdir(directory)

# Make the main figure directory if not present
main_figure_path = directory + sublocation
os.makedirs(main_figure_path, exist_ok=True)

# Define output formats and create their subdirectories
formats = ['pdf', 'jpeg', 'svg', 'html']  # ← added html
for fmt in formats:
    os.makedirs(main_figure_path + fmt + "/", exist_ok=True)

# Save the figure in different formats
for fmt in formats:
    file_path = main_figure_path + fmt + "/" + "_" + name + "." + fmt
    if fmt == "html":
        fig.write_html(file_path)              # ← added HTML saving
    else:
        fig.write_image(file_path, format=fmt, engine="kaleido")
    print(f"Saved {fmt.upper()} to: {file_path}")

# Convert PDF to EPS if needed
# call(["pdf2ps", main_figure_path + "pdf" + "/" + "_" + name + ".pdf", main_figure_path + "eps" + "/" + "_" + name + ".eps"])
